# Step 2 — Ingest to Giant H5

## What this step does

Step 1 produced averaged PSFcam frames matched to PLcam timestamps. Step 2 combines **both cameras** into a single, well-organized H5 file:

- Loads all PLcam FITS files, applies **dark subtraction**, and (optionally) crops to a region of interest (ROI)
- Reads the matched PSFcam frames from Step 1
- Computes a **PSF centroid** (x, y position) and **peak value** (Strehl proxy) for every frame
- Writes everything in time order into one compressed H5 file

```
Step 1 H5  +  PLcam FITS files
                     ↓
          [ ingest_to_h5() ]
                     ↓
           alldata.h5 (giant H5)
           /psfcam/frames      (N, 40, 40)   — averaged PSFcam frames
           /psfcam/centroids   (N, 2)        — PSF centroid (x, y)
           /psfcam/peaks       (N,)          — PSF peak value
           /plcam/frames       (N, ny, nx)   — dark-subtracted PLcam
           /metadata/timestamps (N,)
```

**Why a single H5?** Downstream steps (averaging, extraction) need to quickly access *any* frame by index. One compressed H5 is much faster to work with than hundreds of FITS files.

## Setup: paths and parameters

In [ ]:
import os, sys

TUTORIAL_DIR = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, os.path.join(TUTORIAL_DIR, '..', '..'))

# ── Step 1 output (input to this step) ───────────────────────────────────────
# If you ran tutorial_step1_sort.ipynb, use the new output.
# Otherwise fall back to the pre-existing output.
STEP1_H5_NEW = os.path.join(TUTORIAL_DIR, 'tutorial_output_new', 'step1_fastcam.h5')
STEP1_H5_PRE = os.path.join(TUTORIAL_DIR, 'tutorial_output', 'fastcam.h5')
STEP1_H5 = STEP1_H5_NEW if os.path.exists(STEP1_H5_NEW) else STEP1_H5_PRE
print(f'Using Step 1 H5: {STEP1_H5}')

# ── PLcam calibration and data ───────────────────────────────────────────────
PLCAM_DARK    = os.path.join(TUTORIAL_DIR, 'data', 'slowcam', 'dark.fits')   # dark frame
PLCAM_DIR     = os.path.join(TUTORIAL_DIR, 'data', 'slowcam')                # PLcam FITS files

# ── ROI crop ─────────────────────────────────────────────────────────────────
# (y0, y1, x0, x1) — crop PLcam frames to this pixel region before storing.
# This dramatically reduces file size. Set to None for full frame.
# Here we keep all 412 rows (full height) but only 20 columns around the spectral region.
PLCAM_ROI = (0, 412, 1200, 1220)

# ── Output ───────────────────────────────────────────────────────────────────
OUTPUT_DIR    = os.path.join(TUTORIAL_DIR, 'tutorial_output_new')
OUTPUT_H5     = os.path.join(OUTPUT_DIR, 'step2_alldata.h5')

print(f'PLcam dark : {PLCAM_DARK}')
print(f'PLcam dir  : {PLCAM_DIR}')
print(f'PLcam ROI  : {PLCAM_ROI}  → stored shape per frame: ({PLCAM_ROI[1]-PLCAM_ROI[0]}, {PLCAM_ROI[3]-PLCAM_ROI[2]})')
print(f'Output H5  : {OUTPUT_H5}')

## Peek at the dark frame and PLcam data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

# Dark frame
dark = fits.getdata(PLCAM_DARK).astype('float32')
if dark.ndim == 3:
    dark = dark.mean(axis=0)   # some darks are saved as a cube
print(f'Dark frame shape : {dark.shape}')
print(f'Dark mean        : {dark.mean():.2f}  (detector bias level)')
print(f'Dark std         : {dark.std():.2f}   (readout noise)')

# PLcam FITS files
plcam_fits = sorted(glob.glob(os.path.join(PLCAM_DIR, '*.fits')))
plcam_fits = [f for f in plcam_fits if 'dark' not in os.path.basename(f).lower()]
print(f'\nPLcam FITS files : {len(plcam_fits)}')
for f in plcam_fits:
    print(f'  {os.path.basename(f)}')

# Peek at one PLcam frame
with fits.open(plcam_fits[0]) as hdul:
    raw_frame = hdul[0].data[0].astype('float32')   # first frame in first FITS
print(f'\nOne PLcam frame shape : {raw_frame.shape}')

# Show raw vs dark-subtracted in the ROI
y0, y1, x0, x1 = PLCAM_ROI
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(raw_frame[y0:y1, x0:x1], aspect='auto', origin='lower', cmap='gray')
axes[0].set_title('Raw PLcam (ROI)')
axes[1].imshow((raw_frame - dark)[y0:y1, x0:x1], aspect='auto', origin='lower', cmap='gray')
axes[1].set_title('Dark-subtracted PLcam (ROI)')
plt.tight_layout()
plt.show()

## Run ingest_to_h5

This is the core building block. All parameters are set explicitly above.

In [ ]:
import PLred.ingest as ingest

ingest.ingest_to_h5(
    step1_h5       = STEP1_H5,          # path to Step 1 output
    outpath        = OUTPUT_H5,          # where to write the giant H5
    plcam_dark     = PLCAM_DARK,         # dark FITS for PLcam
    plcam_data_dir = PLCAM_DIR,          # directory with PLcam FITS files
    plcam_roi      = PLCAM_ROI,          # (y0, y1, x0, x1) crop
    verbose        = True,               # print progress per file
)

print('\nStep 2 complete.')

## Inspect the output H5 structure

In [ ]:
import h5py

with h5py.File(OUTPUT_H5, 'r') as f:
    print('=== H5 structure ===')
    def _show(name, obj):
        indent = '  ' * name.count('/')
        if hasattr(obj, 'shape'):
            print(f'{indent}{name}: shape={obj.shape}  dtype={obj.dtype}')
        else:
            print(f'{indent}{name}/')
    f.visititems(_show)

    print('\n=== Root attributes ===')
    for k, v in f.attrs.items():
        print(f'  {k}: {v}')

In [ ]:
import h5py, numpy as np

with h5py.File(OUTPUT_H5, 'r') as f:
    psfcam_frames  = f['psfcam/frames'][:]       # (N, h, w)
    centroids      = f['psfcam/centroids'][:]     # (N, 2)
    peaks          = f['psfcam/peaks'][:]         # (N,)
    plcam_frame0   = f['plcam/frames'][0]         # first PLcam frame
    timestamps     = f['metadata/timestamps'][:]  # (N,) relative seconds

N = len(timestamps)
print(f'N frames           : {N}')
print(f'PSFcam frame shape : {psfcam_frames.shape}')
print(f'PLcam frame shape  : {plcam_frame0.shape}  (after ROI crop)')
print(f'Centroid x range   : [{centroids[:,0].min():.2f}, {centroids[:,0].max():.2f}] pixels')
print(f'Centroid y range   : [{centroids[:,1].min():.2f}, {centroids[:,1].max():.2f}] pixels')
print(f'Peak range         : [{peaks.min():.0f}, {peaks.max():.0f}] counts')
print(f'Duration           : {timestamps[-1]:.2f} seconds')

## Visualize

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# PSFcam first frame
ax = axes[0, 0]
ax.imshow(psfcam_frames[0], origin='lower', cmap='inferno')
ax.set_title('First PSFcam frame')

# PLcam first frame
ax = axes[0, 1]
ax.imshow(plcam_frame0, aspect='auto', origin='lower', cmap='gray',
          vmin=np.percentile(plcam_frame0, 5), vmax=np.percentile(plcam_frame0, 99))
ax.set_title('First PLcam frame (dark-subtracted, ROI)')
ax.set_xlabel('Spectral axis (pixels)')
ax.set_ylabel('Fiber axis (pixels)')

# PSF centroid scatter — this shows how the PSF moved during the observation
ax = axes[1, 0]
sc = ax.scatter(centroids[:, 0], centroids[:, 1],
                c=np.arange(N), cmap='viridis', s=10, alpha=0.7)
plt.colorbar(sc, ax=ax, label='Frame index')
ax.set_xlabel('Centroid x (pixels)')
ax.set_ylabel('Centroid y (pixels)')
ax.set_title('PSF centroid positions over time')
ax.set_aspect('equal')

# Peak values over time — Strehl proxy
ax = axes[1, 1]
ax.plot(timestamps, peaks)
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('PSF peak value (counts)')
ax.set_title('PSF peak over time  (Strehl proxy)')

plt.tight_layout()
plt.show()

## Summary

Step 2 produced `step2_alldata.h5` with:
- **PSFcam data**: frames, centroid positions, peak values
- **PLcam data**: dark-subtracted frames (ROI-cropped)
- **Timestamps**: unified time axis

The centroid scatter shows where the PSF was pointing at each moment — this is the spatial information we will use in Step 3 to bin frames by position.

**Next**: Run `tutorial_step3_average.ipynb` to bin frames by PSF centroid and build coupling maps.

---

## CLI equivalent

The `ingest_to_h5` function maps directly to the `[Ingest]` section of the unified config file:

```ini
[Instrument]
nfib = 38
spectral_orientation = horizontal

[Ingest]
step1_h5       = tutorial_output_new/step1_fastcam.h5
plcam_dark     = data/slowcam/dark.fits
plcam_data_dir = data/slowcam
plcam_roi      = "0,412,1200,1220"
output         = tutorial_output_new/step2_alldata.h5
```

Then:
```python
import PLred.ingest as ingest
ingest.ingest_from_config_unified('obs.ini')
# or:
import PLred.pipeline as pipeline
pipeline.run_mode1('obs.ini', steps=[2])
```

In [ ]:
print('Done. Step 2 output saved to:', OUTPUT_H5)